# Cleaning FMR Data

**Goal:**  
Prepare the Fair Market Rent data so it can be used later in the New Jersey housing affordability analysis.

**Plan:**

1. Load the raw FMR file
2. Filter the data to New Jersey
3. Reshape the data from wide format to county-year format
4. Remove unneeded location columns
5. Fix data types
6. Check for missing values and duplicate county-year rows
7. Save the cleaned FMR dataset

In [71]:
from pathlib import Path
import pandas as pd
import numpy as np

### Step 1: Load the Raw FMR File


In [72]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

fmr_path = project_root / "data" / "FMR" / "FMR_All_1983_2026.csv"
clean_path = project_root / "clean_data"

clean_path.mkdir(exist_ok=True)

fmr_raw = pd.read_csv(fmr_path, dtype=str, encoding="latin1")

fmr_raw.head()

,fips,fips2024,census_region,state,county,cousub,areaname26,name,msa26,fmr26_0,...,fmr85_3,fmr85_4,fmr85,msa83,fmr83_0,fmr83_1,fmr83_2,fmr83_3,fmr83_4,fmr83
0,0100199999,0100199999,3,01,001,99999,"Montgomery, AL MSA",Autauga County,METRO33860M33860,$860,...,344,382,45,5240,186,227,269,332,370,45
1,0100399999,0100399999,3,01,003,99999,"Daphne-Fairhope-Foley, AL MSA",Baldwin County,METRO19300M19300,"$1,094",...,393,439,45,5160,217,257,309,380,425,45
2,0100599999,0100599999,3,01,005,99999,"Barbour County, AL",Barbour County,NCNTY01005N01005,$576,...,387,426,45,10000,212,257,300,374,413,45
3,0100799999,0100799999,3,01,007,99999,"Birmingham-Hoover, AL HUD Metro FMR Area",Bibb County,METRO13820M13820,"$1,024",...,400,447,45,10000,218,265,312,387,433,45
4,0100999999,0100999999,3,01,009,99999,"Birmingham-Hoover, AL HUD Metro FMR Area",Blount County,METRO13820M13820,"$1,024",...,417,462,45,1000,229,280,327,404,448,45


### Step 2: Filter to New Jersey Counties

In [73]:
fmr_nj = fmr_raw[fmr_raw["state"].str.zfill(2) == "34"].copy()
fmr_nj.head()

,fips,fips2024,census_region,state,county,cousub,areaname26,name,msa26,fmr26_0,...,fmr85_3,fmr85_4,fmr85,msa83,fmr83_0,fmr83_1,fmr83_2,fmr83_3,fmr83_4,fmr83
3036,3400199999,3400199999,1,34,001,99999,"Atlantic City-Hammonton, NJ HUD Metro FMR Area",Atlantic County,METRO12100M12100,"$1,346",...,550,615,45,560,269,327,382,477,533,45
3037,3400399999,3400399999,1,34,003,99999,"Bergen-Passaic, NJ HUD Metro FMR Area",Bergen County,METRO35620MM0875,"$1,778",...,677,755,45,875,374,452,531,660,736,45
3038,3400599999,3400599999,1,34,005,99999,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",Burlington County,METRO37980M37980,"$1,397",...,550,615,45,6160,271,327,383,480,532,45
3039,3400799999,3400799999,1,34,007,99999,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",Camden County,METRO37980M37980,"$1,397",...,550,615,45,6160,271,327,383,480,532,45
3040,3400999999,3400999999,1,34,009,99999,"Cape May County, NJ HUD Metro FMR Area",Cape May County,METRO12100N36140,"$1,320",...,550,615,45,560,269,327,382,477,533,45


In [74]:
fmr_nj.shape

(21, 310)

The data only has 21 columns right now because it represents the 21 counties, the years are specified in the columns.

### Step 3: Reshape the FMR Data

Every Row must be a Unique county and year.

In [75]:
years = range(1993, 2014)

id_cols = [
    "state",
    "county",
    "cousub",
    "name",
    "fips",
    "fips2024"
]

fmr_yearly_dfs = []

for year in years:
    yy = str(year)[-2:]
    
    year_cols = [
        f"fmr{yy}_0",
        f"fmr{yy}_1",
        f"fmr{yy}_2",
        f"fmr{yy}_3",
        f"fmr{yy}_4",
        f"fmr{yy}"
    ]
    
    temp = fmr_nj[id_cols + year_cols].copy()
    temp["year"] = year
    
    temp = temp.rename(columns={
        f"fmr{yy}_0": "fmr_0br",
        f"fmr{yy}_1": "fmr_1br",
        f"fmr{yy}_2": "fmr_2br",
        f"fmr{yy}_3": "fmr_3br",
        f"fmr{yy}_4": "fmr_4br",
        f"fmr{yy}": "fmr_percentile"
    })
    
    fmr_yearly_dfs.append(temp)

fmr = pd.concat(fmr_yearly_dfs, ignore_index=True)

fmr.head()

,state,county,cousub,name,fips,fips2024,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,fmr_percentile,year
0,34,001,99999,Atlantic County,3400199999,3400199999,451,552,647,808,909,45,1993
1,34,003,99999,Bergen County,3400399999,3400399999,645,783,929,1161,1301,45,1993
2,34,005,99999,Burlington County,3400599999,3400599999,444,538,634,793,889,45,1993
3,34,007,99999,Camden County,3400799999,3400799999,444,538,634,793,889,45,1993
4,34,009,99999,Cape May County,3400999999,3400999999,451,552,647,808,909,45,1993


In [76]:
fmr.shape

(441, 13)

### Step 4: Remove Unneeded Columns

Some columns are extra location details from the original FMR file. The only location needed is county.

- `state`: all rows are New Jersey
- `county`: only a county code, not the county name
- `cousub`: extra county subdivision detail
- `fips`: extra location detail
- `fips2024`: extra location detail

In [77]:
# Create the 5-digit county FIPS code using state + county code
fmr["county_fips"] = (
    fmr["state"].astype(str).str.zfill(2) +
    fmr["county"].astype(str).str.zfill(3)
)

# Replace the county code with the county name
fmr["county"] = (
    fmr["name"]
    .str.replace(" County", "", regex=False)
    .str.strip()
)

# Remove extra location columns
fmr = fmr.drop(columns=["state", "cousub", "name", "fips", "fips2024"])

# Put columns in the order I want
fmr = fmr[[
    "county_fips",
    "county",
    "year",
    "fmr_0br",
    "fmr_1br",
    "fmr_2br",
    "fmr_3br",
    "fmr_4br",
    "fmr_percentile"
]]

fmr.head()

,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,fmr_percentile
0,34001,Atlantic,1993,451,552,647,808,909,45
1,34003,Bergen,1993,645,783,929,1161,1301,45
2,34005,Burlington,1993,444,538,634,793,889,45
3,34007,Camden,1993,444,538,634,793,889,45
4,34009,Cape May,1993,451,552,647,808,909,45


### Step 5: Fix Data Types

In [78]:
fmr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441 entries, 0 to 440
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   county_fips     441 non-null    object
 1   county          441 non-null    object
 2   year            441 non-null    int64 
 3   fmr_0br         441 non-null    object
 4   fmr_1br         441 non-null    object
 5   fmr_2br         441 non-null    object
 6   fmr_3br         441 non-null    object
 7   fmr_4br         441 non-null    object
 8   fmr_percentile  441 non-null    object
dtypes: int64(1), object(8)
memory usage: 31.1+ KB


The rent columns are showing as `object`, which means pandas is reading them as text. Since these columns are rent values, they need to be changed to numbers. The `county_fips` column can stay as text because it is an ID, not a value for math.

In [79]:
rent_cols = [
    "fmr_0br",
    "fmr_1br",
    "fmr_2br",
    "fmr_3br",
    "fmr_4br",
    "fmr_percentile"
]

for col in rent_cols:
    fmr[col] = pd.to_numeric(fmr[col], errors="coerce")

fmr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441 entries, 0 to 440
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   county_fips     441 non-null    object
 1   county          441 non-null    object
 2   year            441 non-null    int64 
 3   fmr_0br         441 non-null    int64 
 4   fmr_1br         441 non-null    int64 
 5   fmr_2br         441 non-null    int64 
 6   fmr_3br         441 non-null    int64 
 7   fmr_4br         441 non-null    int64 
 8   fmr_percentile  441 non-null    int64 
dtypes: int64(7), object(2)
memory usage: 31.1+ KB


### Step 6: Check for Missing Values and Duplicates


In [80]:
fmr.isnull().sum()

county_fips       0
county            0
year              0
fmr_0br           0
fmr_1br           0
fmr_2br           0
fmr_3br           0
fmr_4br           0
fmr_percentile    0
dtype: int64

In [81]:
fmr.duplicated(subset=["county_fips", "year"]).sum()

np.int64(0)

In [82]:
fmr["year"].value_counts().sort_index()

year
1993    21
1994    21
1995    21
1996    21
1997    21
1998    21
1999    21
2000    21
2001    21
2002    21
2003    21
2004    21
2005    21
2006    21
2007    21
2008    21
2009    21
2010    21
2011    21
2012    21
2013    21
Name: count, dtype: int64

### Step 7: Save the Cleaned FMR Data

In [83]:
fmr.to_csv(clean_path / "FMR_clean.csv", index=False)

In [84]:
fmr.head()

,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,fmr_percentile
0,34001,Atlantic,1993,451,552,647,808,909,45
1,34003,Bergen,1993,645,783,929,1161,1301,45
2,34005,Burlington,1993,444,538,634,793,889,45
3,34007,Camden,1993,444,538,634,793,889,45
4,34009,Cape May,1993,451,552,647,808,909,45
